# 05-7. 약 3GB 웹 로그 스트리밍 분석 실습

## Goal

Apache/Nginx Combined Log 형식의 대용량 접근 로그를 **전체 메모리에 올리지 않고** 분석합니다.

- 한 줄씩 읽고 정규표현식으로 구조화합니다.
- 일정한 배치 크기로 집계해 메모리 사용량을 제한합니다.
- 상태 코드·요청 메서드·상위 IP·상위 경로·시간대별 요청량을 계산합니다.
- 파싱 실패와 분석 품질을 검증합니다.
- 404 집중, 민감 경로 탐색 등 후속 확인 대상을 선별합니다.

> 기본값은 빠르게 실행되는 합성 로그입니다. 실제 약 3GB 로그를 분석하려면 아래 설정 셀에서 `RUN_FULL_DATASET = True`와 `FULL_LOG_PATH`를 지정합니다.

## Setup

### 분석 원칙

1. 원본 로그는 읽기 전용으로 다룹니다.
2. 전체 로그를 하나의 DataFrame으로 만들지 않습니다.
3. 파싱 성공·실패 건수를 모두 집계합니다.
4. 탐지 결과는 침해 확정이 아니라 **조사 우선순위**입니다.
5. IP 주소와 요청 정보는 조직의 개인정보·보존 정책에 따라 취급합니다.

In [1]:
from __future__ import annotations

from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlsplit
import json
import math
import random
import re
import time

import pandas as pd

RUN_FULL_DATASET = False
FULL_LOG_PATH = Path("data/access-3gb.log")
SAMPLE_LOG_PATH = Path("data/sample-access.log")
OUTPUT_DIR = Path("outputs/web-log-analysis")

BATCH_SIZE = 100_000
TOP_N = 20
MALFORMED_SAMPLE_LIMIT = 20

LOG_PATH = FULL_LOG_PATH if RUN_FULL_DATASET else SAMPLE_LOG_PATH
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("분석 모드:", "약 3GB 전체 로그" if RUN_FULL_DATASET else "합성 샘플")
print("입력 경로:", LOG_PATH)

분석 모드: 합성 샘플
입력 경로: data/sample-access.log


### 1. 재현 가능한 샘플 로그 준비

전체 데이터가 없어도 노트북을 처음부터 끝까지 실행할 수 있도록 작은 Combined Log 샘플을 만듭니다. 실제 로그 모드에서는 이 셀을 실행해도 원본 파일을 변경하지 않습니다.

In [2]:
def create_sample_log(path: Path, rows: int = 20_000, seed: int = 443) -> None:
    randomizer = random.Random(seed)
    path.parent.mkdir(parents=True, exist_ok=True)

    normal_ips = [f"192.0.2.{number}" for number in range(1, 81)]
    paths = ["/", "/login", "/products", "/api/items", "/static/app.js"]
    agents = ["Mozilla/5.0", "curl/8.7.1", "SecurityTrainingClient/1.0"]
    start = datetime(2026, 8, 14, tzinfo=timezone.utc)

    with path.open("w", encoding="utf-8", newline="\n") as file:
        for index in range(rows):
            timestamp = start + timedelta(seconds=index * 3)
            ip_address = randomizer.choice(normal_ips)
            request_path = randomizer.choice(paths)
            method = randomizer.choice(["GET", "GET", "GET", "POST"])
            status = randomizer.choice([200, 200, 200, 201, 302, 404])

            # 교육용 이상 징후: 한 IP의 민감 경로 탐색과 반복 404
            if index % 97 == 0:
                ip_address = "198.51.100.44"
                request_path = randomizer.choice([
                    "/.env", "/wp-admin", "/phpmyadmin", "/server-status"
                ])
                status = 404

            size = randomizer.randint(120, 15_000)
            agent = randomizer.choice(agents)
            log_line = (
                f'{ip_address} - - [{timestamp.strftime("%d/%b/%Y:%H:%M:%S %z")}] '
                f'"{method} {request_path} HTTP/1.1" {status} {size} '
                f'"-" "{agent}"\n'
            )
            file.write(log_line)

        file.write('this is an intentionally malformed training line\n')


if not RUN_FULL_DATASET and not SAMPLE_LOG_PATH.exists():
    create_sample_log(SAMPLE_LOG_PATH)

if not LOG_PATH.exists():
    raise FileNotFoundError(
        f"입력 로그가 없습니다: {LOG_PATH}. FULL_LOG_PATH를 확인하세요."
    )

file_size_gb = LOG_PATH.stat().st_size / (1024 ** 3)
print(f"파일 크기: {file_size_gb:.4f} GiB")

파일 크기: 0.0019 GiB


## Steps

### 2. Combined Log 한 줄 파싱

정규표현식은 로그의 구조를 분리하는 데만 사용합니다. 날짜, 상태 코드, 바이트 크기는 이후 Python 자료형으로 변환하고 검증합니다.

In [3]:
COMBINED_LOG_PATTERN = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ '
    r'\[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>[A-Z]+) (?P<target>\S+)(?: HTTP/(?P<http_version>[^"]+))?" '
    r'(?P<status>\d{3}) (?P<bytes>\d+|-) '
    r'"(?P<referrer>[^"]*)" "(?P<user_agent>[^"]*)"$'
)


def parse_combined_log(line: str) -> dict:
    match = COMBINED_LOG_PATTERN.match(line.rstrip("\\n"))
    if match is None:
        raise ValueError("Combined Log 형식과 일치하지 않습니다")

    fields = match.groupdict()
    timestamp = datetime.strptime(
        fields["timestamp"],
        "%d/%b/%Y:%H:%M:%S %z",
    ).astimezone(timezone.utc)

    status = int(fields["status"])
    response_bytes = 0 if fields["bytes"] == "-" else int(fields["bytes"])
    request_path = urlsplit(fields["target"]).path or "/"

    return {
        "ip": fields["ip"],
        "timestamp": timestamp,
        "hour": timestamp.replace(minute=0, second=0, microsecond=0),
        "method": fields["method"],
        "target": fields["target"],
        "path": request_path,
        "status": status,
        "bytes": response_bytes,
        "user_agent": fields["user_agent"],
    }

### 3. 파서 단위 점검

대용량 파일을 처리하기 전에 정상·오류 입력을 작은 범위에서 검증합니다.

In [4]:
valid_example = (
    '203.0.113.10 - - [14/Aug/2026:10:30:00 +0900] '
    '"GET /login?next=%2Fadmin HTTP/1.1" 200 443 "-" "Mozilla/5.0"'
)
parsed_example = parse_combined_log(valid_example)

assert parsed_example["path"] == "/login"
assert parsed_example["status"] == 200
assert parsed_example["bytes"] == 443

try:
    parse_combined_log("broken line")
except ValueError:
    pass
else:
    raise AssertionError("잘못된 로그를 파서가 허용했습니다")

parsed_example

{'ip': '203.0.113.10', 'timestamp': datetime.datetime(2026, 8, 14, 1, 30, tzinfo=datetime.timezone.utc), 'hour': datetime.datetime(2026, 8, 14, 1, 0, tzinfo=datetime.timezone.utc), 'method': 'GET', 'target': '/login?next=%2Fadmin', 'path': '/login', 'status': 200, 'bytes': 443, 'user_agent': 'Mozilla/5.0'}


### 4. 배치 단위 증분 집계

각 배치만 임시로 메모리에 저장하고 `Counter`에 누적합니다. 따라서 파일이 커져도 메모리 사용량은 파일 크기 대신 `BATCH_SIZE`와 고유 키 개수에 주로 좌우됩니다.

In [5]:
SENSITIVE_PATH_PATTERN = re.compile(
    r"(?:^|/)(?:\.env|wp-admin|phpmyadmin|server-status|\.git)(?:/|$)",
    re.IGNORECASE,
)


def new_summary() -> dict:
    return {
        "total_lines": 0,
        "parsed_lines": 0,
        "malformed_lines": 0,
        "total_bytes": 0,
        "status": Counter(),
        "method": Counter(),
        "ip": Counter(),
        "path": Counter(),
        "hour": Counter(),
        "user_agent": Counter(),
        "not_found_by_ip": Counter(),
        "sensitive_path_by_ip": Counter(),
        "malformed_samples": [],
    }


def merge_batch(summary: dict, records: list[dict]) -> None:
    if not records:
        return

    frame = pd.DataFrame.from_records(records)
    summary["parsed_lines"] += len(frame)
    summary["total_bytes"] += int(frame["bytes"].sum())
    summary["status"].update(frame["status"].value_counts().to_dict())
    summary["method"].update(frame["method"].value_counts().to_dict())
    summary["ip"].update(frame["ip"].value_counts().to_dict())
    summary["path"].update(frame["path"].value_counts().to_dict())
    summary["hour"].update(frame["hour"].value_counts().to_dict())
    summary["user_agent"].update(frame["user_agent"].value_counts().to_dict())

    not_found = frame.loc[frame["status"] == 404, "ip"]
    summary["not_found_by_ip"].update(not_found.value_counts().to_dict())

    sensitive_mask = frame["path"].str.contains(
        SENSITIVE_PATH_PATTERN,
        regex=True,
        na=False,
    )
    summary["sensitive_path_by_ip"].update(
        frame.loc[sensitive_mask, "ip"].value_counts().to_dict()
    )


def analyze_log(path: Path, batch_size: int = BATCH_SIZE) -> tuple[dict, float]:
    summary = new_summary()
    batch = []
    started_at = time.perf_counter()

    with path.open("r", encoding="utf-8", errors="replace") as file:
        for line_number, line in enumerate(file, start=1):
            summary["total_lines"] += 1
            try:
                batch.append(parse_combined_log(line))
            except (ValueError, OverflowError) as exc:
                summary["malformed_lines"] += 1
                if len(summary["malformed_samples"]) < MALFORMED_SAMPLE_LIMIT:
                    summary["malformed_samples"].append({
                        "line": line_number,
                        "error": str(exc),
                        "raw": line.rstrip("\n")[:500],
                    })

            if len(batch) >= batch_size:
                merge_batch(summary, batch)
                batch.clear()

    merge_batch(summary, batch)
    elapsed_seconds = time.perf_counter() - started_at
    return summary, elapsed_seconds

### 5. 분석 실행

3GB 파일에서는 저장장치 속도와 로그 형식에 따라 시간이 오래 걸릴 수 있습니다. 처리 중 원본 파일을 수정하지 마세요.

In [6]:
summary, elapsed_seconds = analyze_log(LOG_PATH)

throughput_mb_s = (
    LOG_PATH.stat().st_size / (1024 ** 2) / elapsed_seconds
    if elapsed_seconds > 0 else math.inf
)

print(f"처리 시간: {elapsed_seconds:,.2f}초")
print(f"처리 속도: {throughput_mb_s:,.2f} MiB/s")
print(f"전체 행: {summary['total_lines']:,}")
print(f"정상 행: {summary['parsed_lines']:,}")
print(f"오류 행: {summary['malformed_lines']:,}")

처리 시간: 0.16초
처리 속도: 11.65 MiB/s
전체 행: 20,001
정상 행: 20,000
오류 행: 1


### 6. 핵심 결과 확인

In [7]:
def counter_frame(counter: Counter, key_name: str, top_n: int = TOP_N) -> pd.DataFrame:
    return pd.DataFrame(counter.most_common(top_n), columns=[key_name, "requests"])


status_table = counter_frame(summary["status"], "status", top_n=len(summary["status"]))
method_table = counter_frame(summary["method"], "method", top_n=len(summary["method"]))
top_ips = counter_frame(summary["ip"], "ip")
top_paths = counter_frame(summary["path"], "path")
hourly_requests = counter_frame(summary["hour"], "hour", top_n=len(summary["hour"]))
hourly_requests = hourly_requests.sort_values("hour").reset_index(drop=True)

display(status_table)
display(method_table)
display(top_ips.head(10))
display(top_paths.head(10))
display(hourly_requests.head(10))

   status  requests
0     200      9972
1     404      3496
2     302      3305
3     201      3227
  method  requests
0    GET     15037
1   POST      4963
           ip  requests
0  192.0.2.25       278
1  192.0.2.78       277
2  192.0.2.26       277
3  192.0.2.24       275
4  192.0.2.54       274
5  192.0.2.11       271
6  192.0.2.13       267
7  192.0.2.48       265
8  192.0.2.60       263
9  192.0.2.20       263
             path  requests
0  /static/app.js      4019
1          /login      3966
2       /products      3950
3               /      3934
4      /api/items      3924
5     /phpmyadmin        54
6  /server-status        54
7       /wp-admin        51
8           /.env        48
                       hour  requests
0 2026-08-14 00:00:00+00:00      1200
1 2026-08-14 01:00:00+00:00      1200
2 2026-08-14 02:00:00+00:00      1200
3 2026-08-14 03:00:00+00:00      1200
4 2026-08-14 04:00:00+00:00      1200
5 2026-08-14 05:00:00+00:00      1200
6 2026-08-14 06:00:00+00:00      

### 7. 조사 우선순위 선별

다음 규칙은 교육용 휴리스틱입니다. 높은 요청량이나 반복 404만으로 악성 행위를 확정하지 않습니다. 프록시·NAT·취약점 스캐너·운영 점검 여부와 원본 로그를 함께 확인해야 합니다.

In [8]:
not_found_threshold = 20 if not RUN_FULL_DATASET else 500
sensitive_path_threshold = 3 if not RUN_FULL_DATASET else 20

candidate_ips = set(summary["not_found_by_ip"]) | set(summary["sensitive_path_by_ip"])
triage_rows = []

for ip_address in candidate_ips:
    not_found_count = summary["not_found_by_ip"][ip_address]
    sensitive_count = summary["sensitive_path_by_ip"][ip_address]
    if not_found_count >= not_found_threshold or sensitive_count >= sensitive_path_threshold:
        triage_rows.append({
            "ip": ip_address,
            "total_requests": summary["ip"][ip_address],
            "not_found_404": not_found_count,
            "sensitive_path_requests": sensitive_count,
        })

triage_table = pd.DataFrame(triage_rows)
if not triage_table.empty:
    triage_table = triage_table.sort_values(
        ["sensitive_path_requests", "not_found_404", "total_requests"],
        ascending=False,
    ).reset_index(drop=True)

triage_table.head(20)

               ip  total_requests  not_found_404  sensitive_path_requests
0   198.51.100.44             207            207                      207
1      192.0.2.24             275             64                        0
2       192.0.2.9             238             56                        0
3      192.0.2.21             249             53                        0
4      192.0.2.19             261             52                        0
5      192.0.2.12             252             51                        0
6       192.0.2.3             250             51                        0
7      192.0.2.54             274             50                        0
8      192.0.2.77             244             50                        0
9      192.0.2.49             244             50                        0
10     192.0.2.26             277             49                        0
11     192.0.2.41             260             49                        0
12     192.0.2.57             256     

## Checks

### 8. 데이터 품질과 보존성 검증

전체 행 수는 정상 행과 오류 행의 합과 같아야 합니다. 오류율이 예상보다 높으면 정규표현식을 무작정 완화하지 말고 실제 로그 형식·인코딩·줄바꿈을 먼저 확인합니다.

In [9]:
assert summary["total_lines"] == summary["parsed_lines"] + summary["malformed_lines"]
assert sum(summary["status"].values()) == summary["parsed_lines"]
assert sum(summary["method"].values()) == summary["parsed_lines"]

malformed_rate = (
    summary["malformed_lines"] / summary["total_lines"]
    if summary["total_lines"] else 0
)

quality_report = pd.DataFrame([{
    "file": str(LOG_PATH),
    "size_gib": LOG_PATH.stat().st_size / (1024 ** 3),
    "total_lines": summary["total_lines"],
    "parsed_lines": summary["parsed_lines"],
    "malformed_lines": summary["malformed_lines"],
    "malformed_rate_pct": malformed_rate * 100,
    "elapsed_seconds": elapsed_seconds,
    "throughput_mib_s": throughput_mb_s,
}])

display(quality_report)
display(pd.DataFrame(summary["malformed_samples"]))

                     file  size_gib  ...  elapsed_seconds  throughput_mib_s
0  data/sample-access.log  0.001865  ...         0.163945         11.646886

[1 rows x 8 columns]
    line  ...                                               raw
0  20001  ...  this is an intentionally malformed training line

[1 rows x 3 columns]


### 9. 결과 저장

원본 로그 대신 크기가 작은 집계 결과와 제한된 오류 샘플만 저장합니다. 조사에 필요한 원문 범위는 별도 승인과 보존 정책에 따라 추출합니다.

In [10]:
status_table.to_csv(OUTPUT_DIR / "status_counts.csv", index=False)
method_table.to_csv(OUTPUT_DIR / "method_counts.csv", index=False)
top_ips.to_csv(OUTPUT_DIR / "top_ips.csv", index=False)
top_paths.to_csv(OUTPUT_DIR / "top_paths.csv", index=False)
hourly_requests.to_csv(OUTPUT_DIR / "hourly_requests.csv", index=False)
triage_table.to_csv(OUTPUT_DIR / "triage_candidates.csv", index=False)
quality_report.to_csv(OUTPUT_DIR / "quality_report.csv", index=False)

machine_summary = {
    "input_file": str(LOG_PATH),
    "total_lines": summary["total_lines"],
    "parsed_lines": summary["parsed_lines"],
    "malformed_lines": summary["malformed_lines"],
    "total_response_bytes": summary["total_bytes"],
    "elapsed_seconds": elapsed_seconds,
    "throughput_mib_s": throughput_mb_s,
}

(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(machine_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

sorted(path.name for path in OUTPUT_DIR.iterdir())

['hourly_requests.csv', 'method_counts.csv', 'quality_report.csv', 'status_counts.csv', 'summary.json', 'top_ips.csv', 'top_paths.csv', 'triage_candidates.csv']


## Next Steps

### 실제 약 3GB 로그 실행 절차

1. 원본 로그를 읽기 전용 위치에 준비합니다.
2. 설정 셀에서 `RUN_FULL_DATASET = True`로 변경합니다.
3. `FULL_LOG_PATH`를 실제 파일 경로로 지정합니다.
4. 처음에는 `BATCH_SIZE = 50_000`으로 시작하고 메모리 여유를 확인합니다.
5. 노트북을 위에서 아래로 다시 실행합니다.
6. `quality_report.csv`의 오류율과 처리 건수를 먼저 검증합니다.
7. 후보 IP는 원본 시간 범위, 프록시 구조, 자산 정보와 함께 추가 조사합니다.

### 완료 기준

- [ ] 3GB급 로그를 전체 메모리 적재 없이 처리할 수 있습니다.
- [ ] 파싱 성공·실패 행 수를 검증할 수 있습니다.
- [ ] 상태 코드·메서드·IP·경로·시간대별 요청을 집계할 수 있습니다.
- [ ] 탐지 후보와 침해 확정을 구분할 수 있습니다.
- [ ] 재실행 가능한 CSV·JSON 결과물을 생성할 수 있습니다.